# Serialization & Deserialization

One of the main value props of Pydantic is that it provides first class support for serializing and deserializing model objects.

Deserialization, which is the process of creating a Python object from its representation on disk, is done by -
  * `model_validate()`
  * `model_validate_json()`
  * `model_validate_strings()`
All of which has been already covered in [1_Model_Basics](./1_Model_Basics.ipynb).

Serializaiton, which is the process of saving a Python object on disk, is done by -
  * `model_dump()`
  * `model_dump_json()`

To support advanced ser/der operations I might need access to the model's schema. This is available with `model_json_schema()` function.

For `TypeAdapaters` I need to use `dump_json()` which will return `bytes` which can easily be decoded to `str`

In [2]:
from __future__ import annotations
from pydantic import *  # type: ignore
from datetime import datetime
from pprint import pprint
from enum import StrEnum, auto
from typing import Any, cast

In [3]:
class Cookie(BaseModel):
    flavor: str
    calories: int
    baked_on: datetime

In [4]:
chocolate_chip = Cookie(flavor="Chocolate Chip", calories=200, baked_on=datetime.now())

In [5]:
# Outputs a Python dict
chocolate_chip.model_dump()

{'flavor': 'Chocolate Chip',
 'calories': 200,
 'baked_on': datetime.datetime(2026, 1, 15, 3, 55, 51, 580718)}

Python's `datetime` objects are serialized in full ISO format in JSON.

In [6]:
# Outputs a JSON string
chocolate_chip.model_dump_json()

'{"flavor":"Chocolate Chip","calories":200,"baked_on":"2026-01-15T03:55:51.580718"}'

In [7]:
# I can custoimze the JSON string in a bunch of different ways
print(chocolate_chip.model_dump_json(indent=2))

{
  "flavor": "Chocolate Chip",
  "calories": 200,
  "baked_on": "2026-01-15T03:55:51.580718"
}


In [8]:
schema = chocolate_chip.model_json_schema()
type(schema)

dict

In [9]:
pprint(schema)

{'properties': {'baked_on': {'format': 'date-time',
                             'title': 'Baked On',
                             'type': 'string'},
                'calories': {'title': 'Calories', 'type': 'integer'},
                'flavor': {'title': 'Flavor', 'type': 'string'}},
 'required': ['flavor', 'calories', 'baked_on'],
 'title': 'Cookie',
 'type': 'object'}


In [10]:
class ContentType(StrEnum):
    TEXT = auto()
    VIDEO = auto()
    PODCAST = auto()

In [11]:
class Post(BaseModel):
    content_type: ContentType
    title: str
    published_on: datetime

In [12]:
post = Post(
    title="Should I Just Watch the Sunset?",
    content_type=ContentType.TEXT,
    published_on=datetime(2020, 12, 12),
)

In [13]:
print(post.model_dump_json(indent=2))

{
  "content_type": "text",
  "title": "Should I Just Watch the Sunset?",
  "published_on": "2020-12-12T00:00:00"
}


In [15]:
Cookies = TypeAdapter(list[Cookie])
cookies_data = [
    {"flavor": "Chocolate Chip", "calories": 200, "baked_on": datetime.now()},
    {"flavor": "Snicker Doodle", "calories": 220, "baked_on": datetime.now()},
]
cookies = Cookies.validate_python(cookies_data)
cookies

[Cookie(flavor='Chocolate Chip', calories=200, baked_on=datetime.datetime(2026, 1, 15, 3, 57, 20, 269607)),
 Cookie(flavor='Snicker Doodle', calories=220, baked_on=datetime.datetime(2026, 1, 15, 3, 57, 20, 269609))]

In [23]:
print(Cookies.dump_json(cookies, indent=2).decode())

[
  {
    "flavor": "Chocolate Chip",
    "calories": 200,
    "baked_on": "2026-01-15T03:57:20.269607"
  },
  {
    "flavor": "Snicker Doodle",
    "calories": 220,
    "baked_on": "2026-01-15T03:57:20.269609"
  }
]


## Customize Serialization

This can be useful when I want to output the datetime field or an enum field in some specific format. I can set it at the field level using the `@field_serializer()`, or the model level using the `@model_serializer()` decorators. Custom serialization decorators can also use context data that is outside of the model. At runtime, the context data is provided to the model dumper functions.

Pydantic also has `PlainSerializer` and `WrapSerializer` but I don't think I'll need those any time soon. 

I can use Pydantic's [custom types](https://docs.pydantic.dev/latest/api/types/#pydantic.types) to control how to serialize any specific field.

In [28]:
class User(BaseModel):
    id: int
    created_at: datetime
    bookmarks: list[Bookmark] = []  # type: ignore

    @field_serializer("created_at")
    def serialize_created_at(self, created_at: datetime, _info):
        fmt = "%a, %b %d, %Y | %I:%M %p"
        return created_at.strftime(fmt)


class Bookmark(BaseModel):
    id: int
    added_on: datetime
    url: str
    content_type: ContentType

    @model_serializer
    def ser(self) -> dict[str, Any]:
        fmt = "%a, %b %d, %Y"
        return {
            "id": self.id,
            "added_on": self.added_on.strftime(fmt),
            "url": self.url,
            "content _type": self.content_type.upper(),
        }

In [29]:
user = User(
    id=1,
    created_at=datetime(2024, 5, 10, 18, 23),
    bookmarks=[
        Bookmark(
            id=100,
            added_on=datetime(2024, 5, 10, 19, 0),
            url="https://avilay.github.io/2020/12/12/should-i-just-watch-the-sunset.html",
            content_type=ContentType.TEXT,
        ),
        Bookmark(
            id=200,
            added_on=datetime(2024, 6, 10, 10, 20),
            url="https://youtube.com/playlist?list=PLWv7dKcFUITuj5o29j3Ak2CHG4nlEH-0f&si=wfokyE58E_X0yxyO",
            content_type=ContentType.VIDEO,
        ),
    ],
)
print(user.model_dump_json(indent=2))

{
  "id": 1,
  "created_at": "Fri, May 10, 2024 | 06:23 PM",
  "bookmarks": [
    {
      "id": 100,
      "added_on": "Fri, May 10, 2024",
      "url": "https://avilay.github.io/2020/12/12/should-i-just-watch-the-sunset.html",
      "content _type": "TEXT"
    },
    {
      "id": 200,
      "added_on": "Mon, Jun 10, 2024",
      "url": "https://youtube.com/playlist?list=PLWv7dKcFUITuj5o29j3Ak2CHG4nlEH-0f&si=wfokyE58E_X0yxyO",
      "content _type": "VIDEO"
    }
  ]
}


A nice parameter to the various dumpers is the serialization context. This is additional data that is outside of the model that can be used when serializing it. The documentation has a good example that I am copying here.

In [30]:
class Content(BaseModel):
    text: str

    @field_serializer("text")
    def remove_stopwords(self, text: str, info: SerializationInfo) -> str:
        # The context is whatever the user has supplied to the model dumper
        # There is no way to type annotate it. In this case I just know its type
        if info.context:
            ctx: dict[str, list[str]] = info.context
            stopwords = ctx.get("stopwords", set())
            text = " ".join(
                word for word in text.split() if word.lower() not in stopwords
            )

        return text

In [31]:
content = Content(text="This is an example document")

In [32]:
# no stopwords
content.model_dump()

{'text': 'This is an example document'}

In [33]:
# with stopwords
stopwords = ["this", "is", "an"]
content.model_dump(context={"stopwords": stopwords})

{'text': 'example document'}

I can also use any [custom types](https://docs.pydantic.dev/latest/api/types/#pydantic.types) that control serialization. Note, not all custom types are about serialization. They support a wide variety of use cases, custom serialization is just one of them.

In [62]:
class SystemUser(BaseModel):
    username: str
    password: SecretStr

In [64]:
user = SystemUser(username="aptg", password="password")  # type: ignore

In [65]:
# When serializing, password is masked
user

SystemUser(username='aptg', password=SecretStr('**********'))

In [67]:
user.password

SecretStr('**********')

In [66]:
user.model_dump()

{'username': 'aptg', 'password': SecretStr('**********')}

In [69]:
# I can still get at the underlying value
user.password.get_secret_value()

'password'

In [74]:
# I can use this in conjuction with field_serializer
class AppUser(BaseModel):
    username: str
    password: SecretStr

    @field_serializer("password", when_used="json")
    def ser_password(self, password):
        return password.get_secret_value()

In [75]:
user = AppUser(username="APTG", password="pwd")  # type: ignore

In [76]:
user.model_dump_json()

'{"username":"APTG","password":"pwd"}'

## Pickling

This works as usual without having to do anything special.

In [34]:
import pickle

In [35]:
user_data = pickle.dumps(user)
type(user_data)

bytes

In [36]:
user_data[:20]

b'\x80\x04\x95D\x02\x00\x00\x00\x00\x00\x00\x8c\x08__main_'

In [37]:
pickle.loads(user_data)

User(id=1, created_at=datetime.datetime(2024, 5, 10, 18, 23), bookmarks=[Bookmark(id=100, added_on=datetime.datetime(2024, 5, 10, 19, 0), url='https://avilay.github.io/2020/12/12/should-i-just-watch-the-sunset.html', content_type=<ContentType.TEXT: 'text'>), Bookmark(id=200, added_on=datetime.datetime(2024, 6, 10, 10, 20), url='https://youtube.com/playlist?list=PLWv7dKcFUITuj5o29j3Ak2CHG4nlEH-0f&si=wfokyE58E_X0yxyO', content_type=<ContentType.VIDEO: 'video'>)])

## Alias

My general perception is that that the alias design is very wonky and unintuitive. I'll avoid using it if possible. The only use case that I can think of is when interacting with different systems that follow different naming conventions - specifically snake case and camel case. Here is my recipe for dealing with that.

In [38]:
from pydantic.alias_generators import to_camel

In [39]:
class Person(BaseModel):
    first_name: str
    last_name: str

    model_config = ConfigDict(
        alias_generator=to_camel, validate_by_alias=True, validate_by_name=True
    )

In [45]:
# Initialization works with snake case
ap1 = Person(first_name="Avilay", last_name="Parekh")
ap1

Person(first_name='Avilay', last_name='Parekh')

In [46]:
# Initialization also works with camel case
ap2 = Person(firstName="Avilay", lastName="Parekh")  # type: ignore
ap2

Person(first_name='Avilay', last_name='Parekh')

In [48]:
# Default serialization is with snake case
# does not matter how I initialized the object
print(ap1.model_dump())
print(ap2.model_dump())

{'first_name': 'Avilay', 'last_name': 'Parekh'}
{'first_name': 'Avilay', 'last_name': 'Parekh'}


In [49]:
# I can serialize with camel case
print(ap1.model_dump(by_alias=True))
print(ap2.model_dump(by_alias=True))

{'firstName': 'Avilay', 'lastName': 'Parekh'}
{'firstName': 'Avilay', 'lastName': 'Parekh'}


In [53]:
# Field access is only through snake case
ap1.first_name

'Avilay'

In [54]:
# Field access with camel case does not work
# Read access will throw an AttributeError
try:
    print(ap1.firstName)  # type: ignore
except AttributeError as err:
    pprint(err)

AttributeError("'Person' object has no attribute 'firstName'")


In [55]:
# Write access will throw a ValueError
try:
    ap1.firstName = "APTG"  # type: ignore
except ValueError as err:
    pprint(err)

ValueError('"Person" object has no field "firstName"')
